In [22]:
import pandas as pd
from sqlalchemy import create_engine
import psycopg2

In [23]:
USUARIO = "curso_user"       
SENHA = "curso_password"    
BANCO = "curso_sql"     
HOST = "localhost"
PORTA = "5433"

In [25]:
conexao_url = f"postgresql+psycopg2://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}"
engine = create_engine(conexao_url)
def consertar_acentos(texto):
    if isinstance(texto, str):
        try:
            return texto.encode('latin1').decode('utf8')
        except:
            return texto
    return texto

# RELATÓRIO 1: FATURAMENTO

query_faturamento = """
    SELECT l.codigo AS "Linha", SUM(b.valor_cobrado) AS "Faturamento"
    FROM bilhetagem b
    JOIN viagem v ON b.fk_id_viagem = v.id_viagem
    JOIN horario_linha hl ON v.fk_id_horario = hl.id_horario
    JOIN linha l ON hl.fk_id_linha = l.id_linha
    GROUP BY l.codigo
    ORDER BY "Faturamento" DESC;
"""
df_faturamento = pd.read_sql_query(query_faturamento, con=engine)

# Formatando para Moeda Brasileira
df_faturamento["Faturamento"] = df_faturamento["Faturamento"].apply(lambda x: f"R$ {x:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'))

print("FATURAMENTO TOTAL POR LINHA")
display(df_faturamento)

# RELATÓRIO 2: ESCALA

query_escala = """
    SELECT f.nome AS "Motorista", e.turno AS "Turno", l.codigo AS "Linha"
    FROM funcionario f
    JOIN motorista m ON f.id_funcionario = m.id_funcionario
    JOIN escala e ON m.id_funcionario = e.fk_id_motorista
    JOIN linha l ON e.fk_id_linha = l.id_linha;
"""
df_escala = pd.read_sql_query(query_escala, con=engine)

for col in df_escala.select_dtypes(include=['object']).columns:
    df_escala[col] = df_escala[col].apply(consertar_acentos)

print("ESCALA DOS MOTORISTAS")
display(df_escala)

FATURAMENTO TOTAL POR LINHA


,Linha,Faturamento
0,GAR-101,"R$ 175,50"


ESCALA DOS MOTORISTAS


C:\Users\adrie\AppData\Local\Temp\ipykernel_11452\1880609497.py:41: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_escala.select_dtypes(include=['object']).columns:


,Motorista,Turno,Linha
0,Carlos Alberto Silva,Manha,GAR-101
1,Mariana Costa Ribeiro,Manha,GAR-102
2,Ricardo Souza Santos,Tarde,GAR-201
3,Juliana Almeida Prado,Tarde,CAR-301
4,Marcelo Oliveira Gomes,Noite,CAR-302
5,Fernanda Pereira Lima,Noite,CAR-401
6,Antônio Marcos Alves,Manha,POLO-01
7,Camila Rodrigues Neves,Manha,POLO-02
8,Lucas Gabriel Ferreira,Tarde,JEANS-01
9,Amanda Barbosa Vieira,Tarde,JEANS-02
